In [ ]:
import streamlit as st
import pandas as pd
import snowbooks_extras

In [ ]:
use database RNDC_LAB;
use schema RNDC_LAB.TARGET;

# Data Preview

In [ ]:
select * from DOORDASH_100 limit 10;

In [ ]:
select * from img_tbl limit 10;

# AI_COMPLETE

In [ ]:
SELECT 
    review_id,
    review_text,
    AI_COMPLETE('claude-4-sonnet',
    PROMPT('Give me a 10 word summary of the review {0}', review_text)) AS review_summary
FROM doordash_100
limit 10;

In [ ]:
SELECT 
    img_file,
    AI_COMPLETE('pixtral-large',
    PROMPT('Look at the item {0} and give me a relevant menu item name. Only return the item name.', img_file)) AS menu_item_name,
    AI_COMPLETE('claude-4-sonnet',
    PROMPT('For this item {0} with name {1} give me a catchy menu description. Only return the description.', img_file, menu_item_name)) AS menu_item_description,
FROM img_tbl
limit 10;

# AI_CLASSIFY

In [ ]:
select
    review_id, 
    review_text,
    AI_COMPLETE('claude-4-sonnet',
    PROMPT('Give me a 10 word summary of the review {0}', review_text)) AS review_summary,
    AI_CLASSIFY(review_text,
    ['service', 'ambience', 'quality', 'noise','choices','portions','other'],
    {'output_mode': 'multi'}):labels AS review_theme
from DOORDASH_100
limit 10;

In [ ]:
select
    img_file,
    AI_COMPLETE('pixtral-large',
    PROMPT('Look at the item {0} and give me a relevant menu item name. Only return the item name.', img_file)) AS menu_item_name,
    AI_COMPLETE('claude-4-sonnet',
    PROMPT('For this item {0} with name {1} give me a catchy menu description. Only return the description.', img_file, menu_item_name)) AS menu_item_description,
    AI_CLASSIFY(img_file, ['dessert', 'drink', 'main dish', 'side dish']):labels AS classification
from img_tbl
limit 10;

# AI_FILTER (for both FILTER and JOIN)

In [ ]:
select
    review_id, 
    review_text,
    AI_COMPLETE('claude-4-sonnet',
    PROMPT('Give me a 10 word summary of the review {0}', review_text)) AS review_summary,
    AI_CLASSIFY(review_text,
    ['service', 'ambience', 'quality', 'noise','choices','portions','other'],
    {'output_mode': 'multi'}):labels AS review_theme
from DOORDASH_100
where AI_FILTER(PROMPT('The reviewer liked the portion size: {0}',review_text))
limit 10;

In [ ]:
WITH dish_types AS (
    SELECT 'dessert'   AS dish_type
    UNION ALL
    SELECT 'drink'
    UNION ALL
    SELECT 'main dish'
    UNION ALL
    SELECT 'side dish'
)

select img_tbl.img_file,
dish_type
from img_tbl
join dish_types
on AI_FILTER(PROMPT('Evaluate if this item {0} fits this dish type {1}', img_tbl.img_file, dish_types.dish_type))

# AI_EXTRACT

In [ ]:
select
    review_id, 
    review_text,
    AI_COMPLETE('claude-4-sonnet',
    PROMPT('Give me a 10 word summary of the review {0}', review_text)) AS review_summary,
    AI_CLASSIFY(review_text,
    ['service', 'ambience', 'quality', 'noise','choices','portions','other'],
    {'output_mode': 'multi'}):labels AS review_theme,
    AI_EXTRACT(
             text => review_text,
             responseFormat => {'rating': 'How many stars/rating for the review? Mark as `None` if rating not mentioned in the review', 'menu_item_mentioned': 'What menu item was mentioned? Mark as `None` if item not mentioned in the review'}) as review_extracts
from DOORDASH_100
limit 10;

In [ ]:
WITH extracted_info AS (
  SELECT 
    AI_EXTRACT(
      file => TO_FILE('@RNDC_LAB.TARGET.DOORDASH_IMAGES', 'sci-paper.pdf'),
      responseFormat => [
        ['authors', 'Who are the authors?'],
        ['title', 'What is the document title?'],
        ['correspondence_name', 'Who to correspond with?'],
        ['correspondence_email', 'What is the email address for corrrespondence']
      ]
    ) AS extract_data
)
SELECT
  extract_data:response:authors::VARCHAR AS authors,
  extract_data:response:title::VARCHAR AS title,
  extract_data:response:correspondence_name::VARCHAR AS correspondence_name,
  extract_data:response:correspondence_email::VARCHAR AS correspondence_email
FROM extracted_info;

# AI_SENTIMENT

In [ ]:
select
    review_id, 
    review_text,
    AI_COMPLETE('claude-4-sonnet',
    PROMPT('Give me a 10 word summary of the review {0}', review_text)) AS review_summary,
    AI_CLASSIFY(review_text,
    ['service', 'ambience', 'quality', 'noise','choices','portions','other'],
    {'output_mode': 'multi'}):labels AS review_theme,
    AI_EXTRACT(
             text => review_text,
             responseFormat => {'rating': 'How many stars/rating for the review? Mark as `None` if rating not mentioned in the review', 'menu_item_mentioned': 'What menu item was mentioned? Mark as `None` if item not mentioned in the review'}) as review_extracts,
    AI_SENTIMENT(
        review_text,
        ['service', 'ambience', 'quality', 'noise','choices','portions','value_for_money']
      ) as review_sentiments
from DOORDASH_100,
limit 10;